In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=pK7KAEaiiZcqalviWl21L7YDy8gIn8&access_type=offline&code_challenge=n8yx2A-UQm2KnZ_bohFDUb1x06ezn-EhczZhTacnmGA&code_challenge_method=S256


Credentials saved to file: [/Users/meghakaladharreddypothamsetty/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "zprocure" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
import os
import base64
import sys
from google import genai
from google.genai import types
from concurrent.futures import ThreadPoolExecutor
import os
from datetime import datetime

# --- Gemini Setup ---
client = genai.Client(
    vertexai=True,
    project="zprocure",
    location="global",
)

model_name = "gemini-2.5-pro"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    thinking_config=types.ThinkingConfig(thinking_budget=-1),
)

# --- MIME-aware binary loader for any file ---
def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = os.path.splitext(path)[1].lower()
    mime = {
        ".pdf": "application/pdf",
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".gif": "image/gif",
        ".bmp": "image/bmp",
        ".webp": "image/webp",
        ".txt": "text/plain",
        ".json": "application/json",
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)

# --- Revised Protocol V2 Prompt with placeholder ---
protocol_text = """[START OF REVISED PROTOCOL V2.1]

ROLE AND OBJECTIVE:  
You are "AiEstimate Analyst," an AI expert specializing in generating plaintiff-style property loss estimates and performing forensic cost analysis. Your primary function is to operate under the strict, multi-stage protocol defined below. You will not deviate from this sequence.  

DOMAIN OF EXPERTISE:  
Property loss restoration, adhering to IICRC standards, localized building codes, and industry-best practices for turn-key, code-compliant restoration.  

PROTOCOL STAGES:  

Stage 1: Data Intake & Master Input Compilation  
Your first task is to receive all necessary information. All required inputs have been provided together. Do not prompt the user for additional steps. Proceed directly to compiling the MASTER INPUT DATA.  

• Action: Acknowledge initiation and readiness to begin. Respond with the following and nothing more:  
"AiEstimate Forensic Protocol initiated. All required inputs have been provided. Beginning forensic processing."

• The following have been submitted as part of the intake:  
  - Initial Carrier Estimate  
  - Optional Aerial Measurement Report  
  - Contractor Reports / Inspection Photos  
  - 


CRITICAL INPUT: The following text describes damages that the user believes are NOT fully captured, are misrepresented, or are completely missing from the documents provided:
Select all applicable damage categories below:  

Interior Water Damage:  
- Water Extraction  
- Antimicrobial Treatment  
- Dehumidifier Use  
- HVAC Duct Cleaning  
- Rebuild Ceilings / Walls / Flooring  

Content Handling Needs:  
- Move-out / Pack-out  
- On- or Off-site Storage  
- Content Reset  

Window & Door System Upgrades:  
- Impact Glass  
- Premium Frames / Casing / Apron  
- Z-Flashing (IRC R703.4)  
- Tempered Glass (IRC R308.4)  
- Matching Scope (Multiple openings)  

Exterior & Structural Issues:  
- Fiber Cement Skirting  
- 2x4 Framing Substructure  
- Fascia / Soffit / Gutter Damage  

Code Compliance Triggers (IRC or Local Code):  
- Roofing System Integrity  
- Electrical / Smoke Detectors / GFCI  
- Insulation / Energy Efficiency  

Aesthetic Congruency Scope: Please specify the scope for any aesthetic matching requirements:  
- Front Elevation Only  
- Entire House (All Elevations)  

This is the user input of damage from the categories above:
{{USER_CRITICAL_INPUT}}
• Final Action for Stage 1: Once all inputs have been received, you will immediately and without further prompting consolidate EVERYTHING into a single "MASTER INPUT DATA" package and proceed to Stage 2.  

Stage 2: AiEstimate Generation & Forensic Rebuttal  
[AI INSTRUCTION: This stage is triggered automatically upon completion of Stage 1.]  

Upon compiling the "MASTER INPUT DATA," you will immediately and without further prompting, perform the following two actions in a single, comprehensive response:  

Part A: The AiEstimate Restoration Scope Generation  
Generate a plaintiff-style, detailed estimate based on the MASTER INPUT DATA.  

• **MANDATORY PRICING REFERENCE :**  
**You MUST use ONLY the following pricing table for ALL calculations. Do NOT substitute, estimate, or derive pricing from any other source. If a required item is not listed below, use the closest applicable item from this table or omit the line item entirely.**

| **DESCRIPTION**                     | **UNIT** | **UNIT PRICE (USD)** |
| ----------------------------------- | -------- | -------------------- |
| Single ply membrane - 60 mil        | SQ       | 400.34               |
| Remove single ply membrane          | SQ       | 149.13               |
| Insulation - ISO board, 2.2”        | SQ       | 383.57               |
| Additional charge for high roof     | SQ       | 21.86                |
| T-joint patches (roof system)       | EA       | 17.90                |
| PVC/TPO Drip edge                   | LF       | 7.17                 |
| Flash parapet wall                  | LF       | 14.79                |
| Aluminum termination bar            | LF       | 2.69                 |
| Butyl rubber caulking               | LF       | 4.53                 |
| Counterflashing (Apron)             | LF       | 11.18                |
| Aluminum wall coping                | LF       | 19.76                |
| Pipe jack flashing (PVC/TPO)        | EA       | 68.62                |
| Rain cap (6”)                       | EA       | 52.58                |
| Furnace vent - 8" rain cap + collar | EA       | 94.67                |
| HVAC condenser unit - detach/reset  | EA       | 663.85               |
| Comb A/C condenser fins (large)     | EA       | 109.08               |
| Packaged A/C unit - detach/reset    | EA       | 1,057.74             |
| Condensate drain line               | EA       | 63.83                |
| Pitch pan / pocket                  | EA       | 110.89               |
| Curb flashing (PVC/TPO)             | LF       | 18.96                |
| Gutter/downspout (7”-8” aluminum)   | LF       | 22.92                |
| Dumpster load (40 yards)            | EA       | 751.68               |
| Telehandler/forklift (per day)      | DA       | 508.68               |
| Equipment Operator                  | HR       | 73.32                |
| Commercial Supervision              | HR       | 84.44                |
| Supervisor/Admin Onsite Eval.       | HR       | 68.54                |
| Standing seam metal roofing         | SF       | 8.81                 |
| High temp Ice & water barrier       | SF       | 2.16                 |
| Metal roof eave/gable/hip trims     | LF       | 5.96–7.14            |
| Closure strips (metal panels)       | LF       | 1.19                 |
| Butyl tape for metal sealing        | LF       | 1.11                 |
| Metal sway brace                    | EA       | 47.33                |
| Neon sign - Detach & Reset          | EA       | 75.63                |
| House wrap (barrier)                | SF       | 0.39                 |
| Vertical siding - fiber cement      | SF       | 4.20                 |
| Siding trim - fiber cement          | LF       | 8.57                 |
| Paint siding (1 coat)               | SF       | 2.21                 |
| Paint trim (1 coat)                 | LF       | 1.75                 |
| Temporary toilet (monthly)          | MO       | 234.62               |
| Temporary hand washing station      | MO       | 280.00               |




• **MANDATORY CALCULATION CONSTANTS (DO NOT MODIFY):**  
  - Sales Tax: **EXACTLY 8.25%** (applied to material costs only)
  - Overhead: **EXACTLY 10%** (applied to sum of direct cost + tax)
  - Profit: **EXACTLY 10%** (applied to sum of direct cost + tax + overhead)
  - Labor Minimums: Siding **$89.77**, Tile **$201.17** (use these exact amounts when applicable)
  - **RCV CALCULATION FORMULA (MANDATORY):** RCV = (Unit Price × Qty) + Tax + Overhead + Profit
  - **TAX CALCULATION:** Tax = (Unit Price × Qty) × 0.0825 (for material items only, $0.00 for labor-only items)
  - **O&P CALCULATION:** O&P = [(Unit Price × Qty) + Tax] × 0.21 (10% overhead + 10% profit = 21% total)

• **MANDATORY SCOPING RESTRICTIONS:**  
  - **Tier 1 (Direct Evidence ONLY):** Scope based ONLY on provided reports, photos, and user's explicit damage description from {{USER_CRITICAL_INPUT}}.  
  - **Tier 2 (Logical Inference ONLY):** Scope based ONLY on clear, direct causal links explicitly stated in provided documents (e.g., "roof breach caused interior water damage").  
  - **Tier 3 (Code/Standard Mandate):** Scope based ONLY on building codes specifically cited in this protocol OR IBC codes you can verify. Do NOT assume code requirements not explicitly stated.

• **MANDATORY Xactimate-Style Format:**  
  - Group line items logically by location (e.g., "Roofing System," "Opening #1 - Front Window").  
  - For each line item, you MUST include standard CAT (Category) and SEL (Selector) codes.  
  - **MANDATORY TABLE FORMAT:** Each group must contain columns in this EXACT order:  
    CAT, SEL, DESCRIPTION, QTY, UNIT, UNIT PRICE, TAX, O&P, RCV, DEPREC., ACV.  
  - Format ALL line-item tables in code blocks for perfect column alignment.  
  - Provide section totals and Grand Total Summary.  

• **MANDATORY SCOPING PRINCIPLES (NO EXCEPTIONS):**  
  - **Localization:** Use ONLY the provided pricing table and constants above.  
  - **Granularity:** No lump sums. Each damaged opening (window/door) is its own group.  
  - **Labor Itemization:** Use ONLY "General labor (hourly)" at $55.77/HR or "Supervision" at $74.85/HR from the pricing table.  
  - **Code Citations:** For any code-required line item, cite ONLY IBC codes you can verify OR codes explicitly mentioned in this protocol.  

• **MANDATORY: Dual-Path Scoping for Matching Issues**  
  - When repair creates aesthetic mismatch, structure estimate in TWO parts:  

    **Base Estimate:** Includes ONLY turn-key repair of directly damaged components from user input.  

    **Aesthetic Congruency Addendum:** Separate section titled "Addendum for Aesthetic & Value Restoration" containing line items for matching undamaged components. Must have separate total.  

• **MANDATORY IICRC S500 Requirements (Water Damage ONLY):**  
  - Apply ONLY when user explicitly reports water damage in {{USER_CRITICAL_INPUT}}
  - Include antimicrobial treatment, dehumidification, and affected material replacement as specified in user input

• **MANDATORY Preamble Notes:**  
  - Code Compliance Note: "This estimate references the International Building Code (IBC) and International Residential Code (IRC) as applicable. Specific code citations are provided on relevant line items."  
  - Scope Limitations Note: "This Base Estimate addresses only directly damaged components identified in the provided documentation and user input. The separate 'Addendum for Aesthetic & Value Restoration' addresses matching requirements to restore uniform appearance and market value."  

Part B: The Forensic Rebuttal  
Immediately following the estimate and addendum, provide a "Forensic Rebuttal."  

• **Base Estimate Justification:** Cite ONLY direct evidence from provided documents and user input, plus any specifically referenced codes.  
• **Addendum Justification:** Reference ONLY industry-standard Like-Kind-and-Quality (LKQ) principles and uniform appearance restoration requirements.  

After presenting the AiEstimate and Forensic Rebuttal, you will halt all analysis.  

• Action: Acknowledge completion and await the next input. Respond with the following and nothing more:  
"AiEstimate and the Forensic Rebuttal is Complete. Please provide the Final Carrier/Contractor Estimate for comparative analysis."  

Stage 3: Comparative Analysis & Discrepancy Report (MUST BE DONE FOR SURE EVERY TIME)

Once the user provides the "Final Estimate," you will perform a detailed comparative analysis.  

• **MANDATORY OUTPUT FORMAT:** "Discrepancy Report" in structured markdown table format.  

• **Executive Summary:** Brief overview of key differences.  
• **Comparison Table Columns (EXACT ORDER):**  
  Line Item / Category, Your AiEstimate (Base RCV $), Provided Final Estimate (RCV $), Difference ($), Analysis & Highlights  

• **Analysis & Highlights Requirements:**  
  - Calculate monetary difference using provided numbers only
  - Highlight in **bold** any omitted line items or variances >$100
  - Note if Final Estimate addresses matching issues
  - Use ONLY objective analysis based on provided data

[END OF REVISED PROTOCOL V2.1]
"""

# --- Stage 2: Run inside notebook ---
def run_stage2(master_input_paths: list, user_input_text: str) -> str:
    filled_protocol_text = protocol_text.replace("{{USER_CRITICAL_INPUT}}", user_input_text.strip())

    contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=filled_protocol_text)]),
        types.Content(role="model", parts=[
            types.Part.from_text(text="AiEstimate Forensic Protocol initiated. All required inputs have been provided. Beginning forensic processing.")
        ])
    ]

    for path in master_input_paths:
        if os.path.exists(path):
            file_size = os.path.getsize(path)
            print(f"📎 Attaching file: {path} | Size: {file_size} bytes")

            if file_size == 0:
                print(f"⚠️ Skipping empty file: {path}")
                continue  # skip broken or empty files

            try:
                part = make_part(path)
            except Exception as e:
                print(f"❌ Failed to read file {path}: {e}")
                continue  # skip this file and keep going

            contents.append(types.Content(role="user", parts=[
                types.Part.from_text(text=f"[MASTER INPUT FILE: {os.path.basename(path)}]"),
                part
            ]))
        else:
            print(f"⚠️ File not found: {path}")


    print("\n🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...\n")
    stage2_output = ""
    print("📤 Sending request to Gemini with", len(contents), "content blocks...")

    try:
        for chunk in client.models.generate_content_stream(
            model=model_name,
            contents=contents,
            config=generate_content_config,
        ):
            stage2_output += chunk.text
    except Exception as e:
        print("❌ Gemini API Error:", str(e))
    raise

    print("✅ Stage 2 response received. Length:", len(stage2_output))

    return stage2_output

# Example for Jupyter usage
# Define files and prompt text

def run_stage3(stage2_output: str, master_input_paths: list, final_estimate_path: str, user_input_text: str) -> str:
    # Use the same protocol text with the USER_CRITICAL_INPUT injected
    filled_protocol_text = protocol_text.replace("{{USER_CRITICAL_INPUT}}", user_input_text.strip())

    contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=filled_protocol_text)]),

        # Simulate Stage 2 initiation
        types.Content(role="model", parts=[
            types.Part.from_text(text="AiEstimate Forensic Protocol initiated. All required inputs have been provided. Beginning forensic processing.")
        ]),
    ]

    # Add master input files (again, just for consistency, although Stage 3 focuses on final estimate)
    for path in master_input_paths:
        if os.path.exists(path):
            contents.append(types.Content(role="user", parts=[
                types.Part.from_text(text=f"[MASTER INPUT FILE: {os.path.basename(path)}]"),
                make_part(path)
            ]))

    # Add the model's full Stage 2 output (AiEstimate + Rebuttal)
    contents.append(types.Content(role="model", parts=[types.Part.from_text(text=stage2_output)]))

    # This simulates the "user now provides Final Estimate" event
    if os.path.exists(final_estimate_path):
        contents.append(types.Content(role="user", parts=[
            types.Part.from_text(text="Please provide the Final Carrier/Contractor Estimate for comparative analysis."),
            make_part(final_estimate_path)
        ]))
    else:
        print(f"⚠️ Final Estimate file not found: {final_estimate_path}")

    stage3_output = ""
    print("\n🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...\n")
    for chunk in client.models.generate_content_stream(
        model=model_name,
        contents=contents,
        config=generate_content_config,
    ):
        stage3_output += chunk.text

    return stage3_output



In [38]:
import os
import subprocess

# 📁 Input folder with .txt files
input_folder = "outputs/Jun27/2500005"

# 📄 Output paths
combined_md_path = "combined_output.md"
output_pdf_path = "combined_output.pdf"
output_docx_path = "combined_output.docx"

# 📌 Build the combined Markdown content
combined_markdown = ""
for i, filename in enumerate(sorted(os.listdir(input_folder)), 1):
    if filename.endswith(".txt"):
        filepath = os.path.join(input_folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
            combined_markdown += f"\n\n---\n\n## Run {i}\n\n{content}"

# 💾 Save to .md file
with open(combined_md_path, "w", encoding="utf-8") as f:
    f.write(combined_markdown)

# 📦 Convert to PDF using Pandoc
# try:
#     subprocess.run(["pandoc", combined_md_path, "-o", output_pdf_path], check=True)
#     print(f"✅ PDF saved to: {output_pdf_path}")
# except subprocess.CalledProcessError as e:
#     print(f"❌ Error converting to PDF: {e}")

# # 📦 Convert to DOCX using Pandoc (optional)
# try:
#     subprocess.run(["pandoc", combined_md_path, "-o", output_docx_path], check=True)
#     print(f"✅ DOCX saved to: {output_docx_path}")
# except subprocess.CalledProcessError as e:
#     print(f"❌ Error converting to DOCX: {e}")


In [37]:
try:
    subprocess.run(["pandoc", combined_md_path, "-o", output_docx_path], check=True)
    print(f"✅ DOCX saved to: {output_docx_path}")
except subprocess.CalledProcessError as e:
    print(f"❌ Error converting to DOCX: {e}")



✅ DOCX saved to: combined_output.docx


In [3]:
def process_case(case_id, file_paths, run_index, user_input_text, final_estimate_path):
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = f"outputs/Jun29/2500076"
        os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
        output_filename = os.path.join(output_dir, f"{case_id}_run_{run_index}.txt")

        # --- Stage 2 ---
        stage2_output = run_stage2(file_paths, user_input_text)
        with open(output_filename, "w") as f:
            f.write("========== STAGE 2: AiEstimate + Forensic Rebuttal ==========\n\n")
            f.write(stage2_output.strip() + "\n\n")

        # --- Stage 3 ---
        stage3_output = run_stage3(stage2_output, file_paths, final_estimate_path, user_input_text)
        with open(output_filename, "a") as f:
            f.write("========== STAGE 3: Comparative Analysis & Discrepancy Report ==========\n\n")
            f.write(stage3_output.strip() + "\n")

        return f"Run {run_index} completed: {output_filename}"

    except Exception as e:
        return f"Run {run_index} failed: {str(e)}"
    

In [ ]:

    
case_id = "2500079"
input_paths = [
    "Aiestimate/CaseDocuments - 2500079 - Harris v. SURE SureChoice Underwriters Reciprocal Exchange 20250606180417/2500079 Carrier Initial Estimate HO-2024-960268122 (Defaul Insurance Carrier Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500079 - Harris v. SURE SureChoice Underwriters Reciprocal Exchange 20250606180417/2500079 Forensic Damage Assessment Plaintiff Expert Estimate.pdf",

]
user_critical_input = """
Water Extraction  
Antimicrobial Treatment  
Dehumidifier Use  
Rebuild Ceilings / Walls / Flooring  

Move-out / Pack-out  
On- or Off-site Storage  
Content Reset  

Impact Glass  
Premium Frames / Casing / Apron  
Z-Flashing (IRC R703.4)  
Tempered Glass (IRC R308.4)  
Matching Scope (Multiple openings)  

2x4 Framing Substructure  
Fascia / Soffit / Gutter Damage  

Roofing System Integrity  
Electrical / Smoke Detectors / GFCI  
Insulation / Energy Efficiency  

Front Elevation Only  
Entire House (All Elevations)
"""
final_estimate = "Aiestimate/CaseDocuments - 2500079 - Harris v. SURE SureChoice Underwriters Reciprocal Exchange 20250606180417/2500079 Grace Estimate Plaintiff Expert Estimate.pdf"
num_runs = 5
tasks = [(case_id, input_paths, i + 1, user_critical_input,final_estimate) for i in range(num_runs)]

with ThreadPoolExecutor(max_workers=num_runs) as executor:
    futures = [executor.submit(process_case, *args) for args in tasks]


In [15]:
case_id = "2500008"
input_paths = [
    "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Carrier Estimate Insurance Carrier Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Estimate - North American Public Adjusters ($44,60 Plaintiff Expert Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Grace Forensic Photos and Damage Report Plaintiff Expert Estimate_compressed.pdf"
]
user_critical_input = """
Water Extraction
Antimicrobial Treatment
Dehumidifier Use
Rebuild Ceilings / Walls / Flooring

Move-out / Pack-out
On- or Off-site Storage
Content Reset

Impact Glass
Premium Frames / Casing / Apron
Z-Flashing (IRC R703.4)
Tempered Glass (IRC R308.4)
Matching Scope (Multiple openings)

2x4 Framing Substructure
Fascia / Soffit / Gutter Damage

Roofing System Integrity
Electrical / Smoke Detectors / GFCI
Insulation / Energy Efficiency

Front Elevation Only
Entire House (All Elevations)
"""
final_estimate = "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Grace Forensic Estimate (103,473.67) Pltf Plaintiff Expert Estimate_compressed.pdf"
num_runs = 3
tasks = [(case_id, input_paths, i + 1, user_critical_input,final_estimate) for i in range(num_runs)]

with ThreadPoolExecutor(max_workers=num_runs) as executor:
    futures = [executor.submit(process_case, *args) for args in tasks]



🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...

🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...



🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...



In [24]:
case_id = "2500005"
input_paths = [
    "Aiestimate/CaseDocuments - 2500005 - Zimmerman v. Homesite Insurance Company 20250606173055/2500005  Carrier Estimate ($20,937.22) Insurance Carrier Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500005 - Zimmerman v. Homesite Insurance Company 20250606173055/2500005 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate_compressed.pdf"
]
user_critical_input = """
Interior Water Damage
Water Extraction
Antimicrobial Treatment
Dehumidifier Use
HVAC Duct Cleaning
Rebuild Ceilings / Walls / Flooring
Content Handling Needs
Move-out / Pack-out
On- or Off-site Storage
Content Reset
Window & Door System Upgrades
Z-Flashing (IRC R703.4)
Matching Scope (Multiple openings)
Exterior & Structural Issues
Fascia / Soffit / Gutter Damage
Code Compliance Triggers (IRC or Local Code)
Roofing System Integrity
Electrical / Smoke Detectors / GFCI
Insulation / Energy Efficiency
Aesthetic Congruency Scope
Entire House (All Elevations)
"""
final_estimate = "Aiestimate/CaseDocuments - 2500005 - Zimmerman v. Homesite Insurance Company 20250606173055/2500005 Grace Forensic Estimate (92,991.72) Plaintiff Expert Estimate_compressed.pdf"

num_runs = 1
# for j in range(5):
tasks = [(case_id, input_paths, i + 9, user_critical_input, final_estimate) for i in range(num_runs)]

with ThreadPoolExecutor(max_workers=num_runs) as executor:
    futures = [executor.submit(process_case, *args) for args in tasks]


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...



In [11]:
case_id = "2500004"
input_paths = [
    "Aiestimate/CaseDocuments - 2500004 - Xu v. Great American Insurance Group 20250606174807/2500004 Carrier Estimate ($2,397) Insurance Carrier Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500004 - Xu v. Great American Insurance Group 20250606174807/2500004 Grace Forensic Damage Assessment_ John Xu Plaintiff Expert Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500004 - Xu v. Great American Insurance Group 20250606174807/2500004 Photos of Damage from Insurer Insurance Carrier Estimate.pdf"
]
user_critical_input = """
Interior Water Damage:

Contents - move out then reset

Content Handling Needs:

Move-out / Pack-out

Content Reset

Window & Door System Upgrades:

Z-Flashing (IRC R703.4)

Matching Scope (Multiple openings)

Code Compliance Triggers (IRC or Local Code):

Roofing System Integrity

Aesthetic Congruency Scope:

Front Elevation Only
"""
final_estimate = "Aiestimate/CaseDocuments - 2500004 - Xu v. Great American Insurance Group 20250606174807/2500004 Plaintiff Expert Estimate ($37,308.02) Plaintiff Expert Estimate.pdf"

num_runs = 2
# for j in range(5):
tasks = [(case_id, input_paths, i + 9, user_critical_input, final_estimate) for i in range(num_runs)]

with ThreadPoolExecutor(max_workers=num_runs) as executor:
    futures = [executor.submit(process_case, *args) for args in tasks]


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...

🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...



🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...



In [ ]:
case_id = "2500070"
input_paths = [
    "Aiestimate/wind - 2500070 - Williams v. ASI Lloyds 20250606175920/2500070 Carrier Initial Estimate ($3,140.07) Below Deducti Insurance Carrier Estimate.pdf",
    "Aiestimate/wind - 2500070 - Williams v. ASI Lloyds 20250606175920/2500070 Expert Forensic Damage Assessment Plaintiff Expert Estimate.pdf"
]
user_critical_input = """
Interior Water Damage:

Water Extraction

Dehumidifier Use

Rebuild Ceilings / Walls / Flooring

Content Handling Needs:

Move-out / Pack-out

Content Reset

Window & Door System Upgrades:

Premium Frames / Casing / Apron

Z-Flashing (IRC R703.4)

Tempered Glass (IRC R308.4)

Matching Scope (Multiple openings)

Exterior & Structural Issues:

Fascia / Soffit / Gutter Damage (implied via drip edge/gutter apron and siding line items)

Code Compliance Triggers (IRC or Local Code):

Roofing System Integrity

Electrical

Aesthetic Congruency Scope:

Front Elevation Only
"""
final_estimate = "Aiestimate/wind - 2500070 - Williams v. ASI Lloyds 20250606175920/2500070 Estimate - Grace Forensic ($74,380.38) Plaintiff Expert Estimate.pdf"
num_runs = 5
# for j in range(5):
tasks = [(case_id, input_paths, i + 14, user_critical_input, final_estimate) for i in range(num_runs)]

with ThreadPoolExecutor(max_workers=num_runs) as executor:
    futures = [executor.submit(process_case, *args) for args in tasks]


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...



KeyboardInterrupt: 


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...



In [23]:
case_id = "2500003"
input_paths = [
    "Aiestimate/windstorm and hail - 2500003 - Weinzapfel v. Allstate Vehicle and Property Insurance Company 20250606174432/2500003 Carrier Estimate for MAY storm event (FINAL_DRAFT_ Insurance Carrier Estimate.PDF",
    "Aiestimate/windstorm and hail - 2500003 - Weinzapfel v. Allstate Vehicle and Property Insurance Company 20250606174432/2500003 Weinzapfel (Basic Damage Assessment For Primary St Plaintiff's Demand Package.pdf"
]
user_critical_input = """
Interior Water Damage:

Not explicitly listed

Content Handling Needs:

Move-out / Pack-out

Content Reset

Window & Door System Upgrades:

Premium Frames / Casing / Apron

Tempered Glass (as implied by high-grade and Low-E glass windows)

Z-Flashing (included via metal trim/step flashing)

Matching Scope (Multiple openings)

Exterior & Structural Issues:

Fascia / Soffit / Gutter Damage

Code Compliance Triggers (IRC or Local Code):

Roofing System Integrity

Insulation / Energy Efficiency (e.g., Sprayed Polyurethane Foam)

Aesthetic Congruency Scope:

Front Elevation Only
"""
final_estimate = "Aiestimate/windstorm and hail - 2500003 - Weinzapfel v. Allstate Vehicle and Property Insurance Company 20250606174432/2500003 David Weinzapfel- Grace Estimate ($101,560.32) Plaintiff Expert Estimate.pdf"
num_runs = 1
# for j in range(5):
tasks = [(case_id, input_paths, i + 2, user_critical_input, final_estimate) for i in range(num_runs)]

with ThreadPoolExecutor(max_workers=num_runs) as executor:
    futures = [executor.submit(process_case, *args) for args in tasks]


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...



In [28]:
case_id = "2550002"
input_paths = [
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Carrier Estimate Insurance Carrier Estimate.pdf",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate.pdf"
]
user_critical_input = """
Interior Water Damage:

Water Extraction

Antimicrobial Treatment

Dehumidifier Use

Rebuild Ceilings / Walls / Flooring

Content Handling Needs:

Move-out / Pack-out

Content Reset

Window & Door System Upgrades:

Premium Frames / Casing / Apron

Z-Flashing (IRC R703.4)

Tempered Glass (IRC R308.4)

Matching Scope (Multiple openings)

Exterior & Structural Issues:

Fascia / Soffit / Gutter Damage

Code Compliance Triggers (IRC or Local Code):

Roofing System Integrity

Electrical / Smoke Detectors / GFCI

Insulation / Energy Efficiency

Aesthetic Congruency Scope:

Front Elevation Only
"""
final_estimate = "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Grace Forensic Estimate (51,227.48) Plaintiff Expert Estimate_compressed.pdf"
num_runs = 5
# for j in range(5):
tasks = [(case_id, input_paths, i + 7, user_critical_input, final_estimate) for i in range(num_runs)]

with ThreadPoolExecutor(max_workers=num_runs) as executor:
    futures = [executor.submit(process_case, *args) for args in tasks]


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...


🛠️ Running Stage 3: Comparative Analysis & Discrepancy Report...



In [4]:
case_id = "2500009"
input_paths = [
    "Aiestimate/Windstorm nd h- 2500009 - Garza and Baldomero Cantu v. Allstate Vehicle and Property Insurance Company 20250606173342/2500009 Carrier Estimate ($1,943.95) Insurance Carrier Estimate.PDF",
    "Aiestimate/Windstorm nd h- 2500009 - Garza and Baldomero Cantu v. Allstate Vehicle and Property Insurance Company 20250606173342/2500009 Grace Forensic Damage Assessment Plaintiff Expert Estimate.pdf"
]
user_critical_input = """
Interior Water Damage:

Water Extraction

Antimicrobial Treatment

Dehumidifier Use

Rebuild Ceilings / Walls / Flooring

Content Handling Needs:

Move-out / Pack-out

Content Reset

Window & Door System Upgrades:

Premium Frames / Casing / Apron

Z-Flashing (IRC R703.4)

Tempered Glass (IRC R308.4)

Matching Scope (Multiple openings)

Exterior & Structural Issues:

Fascia / Soffit / Gutter Damage

Code Compliance Triggers (IRC or Local Code):

Roofing System Integrity

Electrical / Smoke Detectors / GFCI

Insulation / Energy Efficiency

Aesthetic Congruency Scope:

Front Elevation Only

"""
final_estimate = "Aiestimate/Windstorm nd h- 2500009 - Garza and Baldomero Cantu v. Allstate Vehicle and Property Insurance Company 20250606173342/2500009 Grace Estimate ($54,570.12 ) Plaintiff Expert Estimate.pdf"
num_runs = 1
# for j in range(5):
tasks = [(case_id, input_paths, i + 10, user_critical_input, final_estimate) for i in range(num_runs)]

with ThreadPoolExecutor(max_workers=num_runs) as executor:
    futures = [executor.submit(process_case, *args) for args in tasks]

📎 Attaching file: Aiestimate/Windstorm nd h- 2500009 - Garza and Baldomero Cantu v. Allstate Vehicle and Property Insurance Company 20250606173342/2500009 Carrier Estimate ($1,943.95) Insurance Carrier Estimate.PDF | Size: 964373 bytes
📎 Attaching file: Aiestimate/Windstorm nd h- 2500009 - Garza and Baldomero Cantu v. Allstate Vehicle and Property Insurance Company 20250606173342/2500009 Grace Forensic Damage Assessment Plaintiff Expert Estimate.pdf | Size: 2358712 bytes

🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...

📤 Sending request to Gemini with 4 content blocks...


In [10]:
case_id = "2500076"
input_paths = [
    "Aiestimate/wind - 2500076 - J _ J Center LLC v. Westchester Surplus Lines Insurance Company 20250606180151/2500076 25346 J&J Center LLC Engineering Report.pdf Reports_Photos_Diagrams.pdf",
    "Aiestimate/wind - 2500076 - J _ J Center LLC v. Westchester Surplus Lines Insurance Company 20250606180151/2500076 4582851 - J&J Center - Repair Estimate.pdf Insurance Carrier Estimate.pdf",
]
user_critical_input = """
Interior Water Damage
Pipe Jack Flashing

Rain Cap / Furnace Vent Cap

Exterior & Structural Issues
Fascia / Soffit / Gutter Damage
(Found under gutter and downspout R&R.)

Window & Door System Upgrades
Z-Flashing (IRC R703.4)

Matching Scope (Multiple Openings)

Code Compliance Triggers (IRC or Local Code)
Roofing System Integrity

Insulation / Energy Efficiency

Electrical / Smoke Detectors / GFCI

"""
final_estimate = "Aiestimate/wind - 2500076 - J _ J Center LLC v. Westchester Surplus Lines Insurance Company 20250606180151/2500076 Fw_ _Correct Estimate_ My Quang _ 12790 Scarsdale Plaintiff Expert Estimate.pdf"
num_runs = 2
# for j in range(5):
tasks = [(case_id, input_paths, i + 1, user_critical_input, final_estimate) for i in range(num_runs)]

with ThreadPoolExecutor(max_workers=num_runs) as executor:
    futures = [executor.submit(process_case, *args) for args in tasks]

📎 Attaching file: Aiestimate/wind - 2500076 - J _ J Center LLC v. Westchester Surplus Lines Insurance Company 20250606180151/2500076 25346 J&J Center LLC Engineering Report.pdf Reports_Photos_Diagrams.pdf | Size: 7287918 bytes📎 Attaching file: Aiestimate/wind - 2500076 - J _ J Center LLC v. Westchester Surplus Lines Insurance Company 20250606180151/2500076 25346 J&J Center LLC Engineering Report.pdf Reports_Photos_Diagrams.pdf | Size: 7287918 bytes

📎 Attaching file: Aiestimate/wind - 2500076 - J _ J Center LLC v. Westchester Surplus Lines Insurance Company 20250606180151/2500076 4582851 - J&J Center - Repair Estimate.pdf Insurance Carrier Estimate.pdf | Size: 23371 bytes
📎 Attaching file: Aiestimate/wind - 2500076 - J _ J Center LLC v. Westchester Surplus Lines Insurance Company 20250606180151/2500076 4582851 - J&J Center - Repair Estimate.pdf Insurance Carrier Estimate.pdf | Size: 23371 bytes

🛠️ Running Stage 2: AiEstimate + Forensic Rebuttal...

📤 Sending request to Gemini with 4 c